In [1]:
# Install if needed
!pip install tensorflow

# Imports
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
print("Loading IMDB reviews dataset...")

VOCAB_SIZE = 12000
MAX_SEQUENCE_LEN = 180

(train_reviews, train_labels), (test_reviews, test_labels) = imdb.load_data(num_words=VOCAB_SIZE)

# Padding
train_reviews = pad_sequences(train_reviews, maxlen=MAX_SEQUENCE_LEN, padding='post')
test_reviews = pad_sequences(test_reviews, maxlen=MAX_SEQUENCE_LEN, padding='post')

print("Training samples:", train_reviews.shape)

Loading IMDB reviews dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: (25000, 180)


In [3]:
X_train_t = torch.LongTensor(train_reviews)
y_train_t = torch.FloatTensor(train_labels)

X_test_t = torch.LongTensor(test_reviews)
y_test_t = torch.FloatTensor(test_labels)

BATCH_SIZE = 128

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [4]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, filters=128, kernel_size=5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=filters,
            kernel_size=kernel_size
        )

        self.activation = nn.ReLU()
        self.classifier = nn.Linear(filters, 1)

    def forward(self, x):
        x = self.embedding(x)

        # Change shape for Conv1D → (batch, channels, seq_len)
        x = x.permute(0, 2, 1)

        x = self.activation(self.conv(x))

        # Global max pooling
        x = torch.max(x, dim=2)[0]

        out = self.classifier(x)

        return out.squeeze()

In [5]:
model = TextCNN(VOCAB_SIZE).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 6

In [6]:
print("Training model...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    acc = correct / len(train_loader.dataset)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {acc*100:.2f}%")

Training model...

Epoch 1 | Loss: 0.5425 | Accuracy: 72.49%
Epoch 2 | Loss: 0.3460 | Accuracy: 85.41%
Epoch 3 | Loss: 0.2341 | Accuracy: 91.68%
Epoch 4 | Loss: 0.1471 | Accuracy: 95.88%
Epoch 5 | Loss: 0.0835 | Accuracy: 98.75%
Epoch 6 | Loss: 0.0446 | Accuracy: 99.82%


In [7]:
model.eval()

total_loss = 0
correct = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct += (preds == labels).sum().item()

final_loss = total_loss / len(test_loader)
final_acc = correct / len(test_loader.dataset)

print("\nFinal Results:")
print(f"Test Loss: {final_loss:.4f}")
print(f"Test Accuracy: {final_acc*100:.2f}%")


Final Results:
Test Loss: 0.3314
Test Accuracy: 86.56%
